<a href="https://colab.research.google.com/github/nagasivaninandam/Advanced-Data-Science-Internship-Case-Studies/blob/master/movie_recommendation_system_using_the_MovieLens_100K_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Setup and Data Loading

First, we need to import the necessary libraries and load the data files from Google Drive.  
The main files we'll use are:

- `u.data` → contains user ratings
- `u.item` → contains movie titles

We'll also make sure the dataset is properly loaded into pandas DataFrames.

In [1]:
import pandas as pd
import numpy as np

# Define the file paths based on your screenshot
base_path = '/content/drive/My Drive/AI Projects/ml-100k/'
ratings_path = base_path + 'u.data'
movies_path = base_path + 'u.item'

# Load the ratings data
# It's a tab-separated file with columns: user_id, item_id, rating, timestamp
ratings_cols = ['user_id', 'item_id', 'rating', 'timestamp']
ratings = pd.read_csv(ratings_path, sep='\t', names=ratings_cols)

# Load the movie data
# It's a pipe-separated file, we only need the first two columns: item_id and movie_title
movies_cols = ['item_id', 'movie_title']
movies = pd.read_csv(movies_path, sep='|', names=movies_cols, usecols=range(2), encoding='latin-1')

# Merge the two dataframes into one for easier analysis
data = pd.merge(ratings, movies, on='item_id')

# Let's look at the first few rows
print(data.head())

   user_id  item_id  rating  timestamp                 movie_title
0      196      242       3  881250949                Kolya (1996)
1      186      302       3  891717742    L.A. Confidential (1997)
2       22      377       1  878887116         Heavyweights (1994)
3      244       51       2  880606923  Legends of the Fall (1994)
4      166      346       1  886397596         Jackie Brown (1997)


# 2. Data Preparation

Before building the model, it's good practice to understand the data.  
We'll create a new dataframe that shows the **average rating** and the **number of ratings** for each movie.  

This helps us filter out movies with very few reviews, which can skew the recommendations.


In [2]:
# Create a new dataframe with the average rating and number of ratings for each movie
ratings_summary = pd.DataFrame(data.groupby('movie_title')['rating'].mean())
ratings_summary['number_of_ratings'] = data.groupby('movie_title')['rating'].count()

print(ratings_summary.head())

                             rating  number_of_ratings
movie_title                                           
'Til There Was You (1997)  2.333333                  9
1-900 (1994)               2.600000                  5
101 Dalmatians (1996)      2.908257                109
12 Angry Men (1957)        4.344000                125
187 (1997)                 3.024390                 41


# 3. Building the Recommendation Logic

The core of our recommender is a **user-item matrix**.  

- Rows represent **users**  
- Columns represent **movies**  
- Values are the **ratings**  

We can then calculate the **Pearson correlation** between movies.  
A high correlation between two movies means that users who liked one movie also tended to like the other.


In [3]:
# Create the user-item matrix using a pivot table
moviemat = data.pivot_table(index='user_id', columns='movie_title', values='rating')

# Let's see the first few rows of the matrix
print(moviemat.head())

def get_recommendations(movie_title, min_ratings=100):
    """
    Finds movies similar to a given movie title.

    Args:
        movie_title (str): The name of the movie to get recommendations for.
        min_ratings (int): The minimum number of ratings a movie must have to be recommended.

    Returns:
        pandas.DataFrame: A DataFrame of recommended movies, sorted by correlation.
    """
    # Get all the ratings for the specified movie
    movie_user_ratings = moviemat[movie_title]

    # Calculate the correlation between this movie and all other movies
    similar_to_movie = moviemat.corrwith(movie_user_ratings)

    # Create a dataframe of the correlation results
    corr_movie = pd.DataFrame(similar_to_movie, columns=['Correlation'])
    corr_movie.dropna(inplace=True)

    # Join the correlation data with the number of ratings
    corr_movie = corr_movie.join(ratings_summary['number_of_ratings'])

    # Filter out movies that have less than the minimum number of ratings
    recommendations = corr_movie[corr_movie['number_of_ratings'] > min_ratings].sort_values('Correlation', ascending=False)

    return recommendations

movie_title  'Til There Was You (1997)  1-900 (1994)  101 Dalmatians (1996)  \
user_id                                                                       
1                                  NaN           NaN                    2.0   
2                                  NaN           NaN                    NaN   
3                                  NaN           NaN                    NaN   
4                                  NaN           NaN                    NaN   
5                                  NaN           NaN                    2.0   

movie_title  12 Angry Men (1957)  187 (1997)  2 Days in the Valley (1996)  \
user_id                                                                     
1                            5.0         NaN                          NaN   
2                            NaN         NaN                          NaN   
3                            NaN         2.0                          NaN   
4                            NaN         NaN                 

# 4. Get Your Recommendations!

Now we can use our logic to get recommendations for **any movie** in the dataset.  

Let's try getting recommendations for:
- `"Star Wars (1977)"`
- `"Toy Story (1995)"`

We'll use the movie correlation data we calculated earlier and filter out movies with very few ratings.


In [4]:
# Get recommendations for 'Star Wars (1977)'
recommendations_sw = get_recommendations('Star Wars (1977)', min_ratings=100)
print("Recommendations for 'Star Wars (1977)':")
print(recommendations_sw.head())

print("\n" + "="*50 + "\n")

# Get recommendations for 'Toy Story (1995)'
recommendations_ts = get_recommendations('Toy Story (1995)', min_ratings=100)
print("Recommendations for 'Toy Story (1995)':")
print(recommendations_ts.head())

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


Recommendations for 'Star Wars (1977)':
                                                    Correlation  \
movie_title                                                       
Star Wars (1977)                                       1.000000   
Empire Strikes Back, The (1980)                        0.747981   
Return of the Jedi (1983)                              0.672556   
Raiders of the Lost Ark (1981)                         0.536117   
Austin Powers: International Man of Mystery (1997)     0.377433   

                                                    number_of_ratings  
movie_title                                                            
Star Wars (1977)                                                  583  
Empire Strikes Back, The (1980)                                   367  
Return of the Jedi (1983)                                         507  
Raiders of the Lost Ark (1981)                                    420  
Austin Powers: International Man of Mystery (1997)        

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Recommendations for 'Toy Story (1995)':
                               Correlation  number_of_ratings
movie_title                                                  
Toy Story (1995)                  1.000000                452
Craft, The (1996)                 0.549100                104
Down Periscope (1996)             0.457995                101
Miracle on 34th Street (1994)     0.456291                101
G.I. Jane (1997)                  0.454756                175


# Understanding the Recommendations Output

The output shows the **top recommended movies** for the selected titles, along with their **correlation** to the chosen movie and the **number of ratings** each movie has received.

### Key Points:

1. **Correlation**  
   - This measures how similar the movie is to the chosen movie based on user ratings.  
   - A correlation of **1.0** means the movie is perfectly correlated with itself.  
   - Higher values indicate stronger similarity.

2. **Number of Ratings**  
   - Shows how many users have rated the movie.  
   - Movies with very few ratings may be less reliable, so our filtering ensures we only include popular movies.

### Example:

- For `"Star Wars (1977)"`, the top recommendations are:
  - `"Empire Strikes Back, The (1980)"` (correlation ~0.75, 367 ratings)
  - `"Return of the Jedi (1983)"` (correlation ~0.67, 507 ratings)

- For `"Toy Story (1995)"`, the top recommendations are:
  - `"Craft, The (1996)"` (correlation ~0.55, 104 ratings)
  - `"Down Periscope (1996)"` (correlation ~0.46, 101 ratings)

This means users who liked the chosen movie also tended to like these recommended movies.  

> ⚠️ The warnings from NumPy (e.g., divide by zero or degrees of freedom) are expected when computing correlations for movies with very few overlapping ratings. They do **not affect** the main recommendations for popular movies.
